# Curadoria — Pipeline FALSO (Google Fact Check)

Lê todos os CSVs raw do pipeline `pipeline_falso_google_factcheck/raw/`,
acumula o histórico, aplica limpeza e padronização e salva um CSV curated com timestamp.

**Regras desta camada:**
- Remover HTML do `texto_afirmacao`
- Normalizar espaços e capitalização
- Padronizar datas para `YYYY-MM-DD`
- Remover duplicatas por `texto_principal + fonte + url_origem`
- Mapear `avaliacao_original` para categoria normalizada
- Adicionar colunas do schema obrigatório curated
- **Não** modificar arquivos raw
- **Não** gerar dataset final de treino

## Bibliotecas

In [1]:
import re
import uuid
import pandas as pd

from datetime import datetime
from pathlib import Path
from html.parser import HTMLParser

## Configuração de caminhos

In [2]:
NOME_PIPELINE = "pipeline_falso_google_factcheck"

PASTA_RAW     = Path(f"../dados/{NOME_PIPELINE}/raw")
PASTA_CURATED = Path(f"../dados/{NOME_PIPELINE}/curated")
PASTA_CURATED.mkdir(parents=True, exist_ok=True)

print(f"Raw:     {PASTA_RAW}")
print(f"Curated: {PASTA_CURATED}")

Raw:     ..\dados\pipeline_falso_google_factcheck\raw
Curated: ..\dados\pipeline_falso_google_factcheck\curated


## Funções utilitárias

In [ ]:
class _StripHTML(HTMLParser):
    """Parser simples que descarta tags e acumula apenas o texto."""
    def __init__(self):
        super().__init__()
        self._partes = []

    def handle_data(self, data):
        self._partes.append(data)

    def get_text(self):
        return " ".join(self._partes)


def remover_html(texto: str) -> str:
    """Remove tags HTML e normaliza espaços em branco."""
    if not isinstance(texto, str) or not texto.strip():
        return ""
    parser = _StripHTML()
    parser.feed(texto)
    limpo = parser.get_text()
    limpo = re.sub(r"\s+", " ", limpo).strip()
    return limpo


def padronizar_data(valor) -> str:
    """
    Tenta converter datas em vários formatos para YYYY-MM-DD.
    Retorna string vazia se não conseguir.
    """
    if not isinstance(valor, str) or not valor.strip():
        return ""
    # Remove sufixo de hora ISO se presente: 2026-05-05T00:00:00Z → 2026-05-05
    valor = valor.strip()
    formatos = [
        "%Y-%m-%dT%H:%M:%SZ",
        "%Y-%m-%dT%H:%M:%S",
        "%Y-%m-%d",
        "%d/%m/%Y",
        "%d/%m/%Y %H:%M:%S",
    ]
    for fmt in formatos:
        try:
            return datetime.strptime(valor, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return ""


# Mapeamento de avaliação original → categoria normalizada
# Termos variados que as organizações de fact-checking usam
_MAPA_AVALIACAO = {
    # Falso direto
    "falso": "FALSO",
    "false": "FALSO",
    "incorreto": "FALSO",
    "incorrect": "FALSO",
    "mentira": "FALSO",
    # Enganoso
    "enganoso": "ENGANOSO",
    "misleading": "ENGANOSO",
    "distorcido": "ENGANOSO",
    "parcialmente falso": "ENGANOSO",
    "mostly false": "ENGANOSO",
    "half true": "ENGANOSO",
    # Impreciso / sem contexto
    "fora de contexto": "FORA_DE_CONTEXTO",
    "sem contexto": "FORA_DE_CONTEXTO",
    "out of context": "FORA_DE_CONTEXTO",
    "impreciso": "IMPRECISO",
    "exagerado": "IMPRECISO",
    # Não verificável
    "não verificável": "NAO_VERIFICAVEL",
    "unverified": "NAO_VERIFICAVEL",
    # Verdadeiro / confirmado pelo fact-checker
    # Não incluir: "explica", "contextualizando", "sem registro" — ambíguos → OUTRO
    "verdadeiro": "VERDADEIRO",
    "true": "VERDADEIRO",
    "comprovado": "VERDADEIRO",
    "certo": "VERDADEIRO",
    "correto": "VERDADEIRO",
    "fato": "VERDADEIRO",
    "fato verificado": "VERDADEIRO",
    "confirmado": "VERDADEIRO",
}


def mapear_avaliacao(valor: str) -> str:
    """Normaliza a avaliação original para uma categoria padronizada."""
    if not isinstance(valor, str) or not valor.strip():
        return "NAO_CLASSIFICADO"
    chave = valor.strip().lower()
    return _MAPA_AVALIACAO.get(chave, "OUTRO")


print("Funções utilitárias definidas.")

## Leitura de todos os CSVs raw

> Arquivos com `_TESTE_` no nome são ignorados automaticamente — eles são gerados em MODO_TESTE e não devem entrar na curadoria oficial.

In [4]:
todos_csvs   = sorted(PASTA_RAW.glob("*.csv"))
arquivos_raw = [a for a in todos_csvs if "_TESTE_" not in a.name]
ignorados    = [a for a in todos_csvs if "_TESTE_" in a.name]

if ignorados:
    print(f"Arquivos _TESTE_ ignorados ({len(ignorados)}):")
    for arq in ignorados:
        print(f"  (ignorado) {arq.name}")

print(f"\nArquivos raw oficiais encontrados: {len(arquivos_raw)}")
for arq in arquivos_raw:
    print(f"  {arq.name}")

if not arquivos_raw:
    raise FileNotFoundError("Nenhum arquivo raw oficial encontrado em " + str(PASTA_RAW))

frames = []
for arq in arquivos_raw:
    df_arq = pd.read_csv(arq, encoding="utf-8-sig", dtype=str)
    df_arq["arquivo_raw_origem"] = arq.name
    frames.append(df_arq)

df_raw = pd.concat(frames, ignore_index=True)
print(f"\nTotal bruto acumulado: {len(df_raw)} registros")
df_raw.head(3)

Arquivos _TESTE_ ignorados (1):
  (ignorado) google_factcheck_raw_TESTE_2026-05-16_21-55-45.csv

Arquivos raw oficiais encontrados: 7
  google_factcheck_raw.csv
  google_factcheck_raw_2026-04-30_00-58-54.csv
  google_factcheck_raw_2026-05-09_15-26-18.csv
  google_factcheck_raw_2026-05-09_15-32-24.csv
  google_factcheck_raw_2026-05-10_02-42-39.csv
  google_factcheck_raw_2026-05-12_23-02-04.csv
  google_factcheck_raw_2026-05-17_02-42-35.csv

Total bruto acumulado: 5726 registros


,termo_busca,texto_afirmacao,data_claim,fonte,url_checagem,avaliacao_original,data_publicacao,arquivo_raw_origem,fonte_verificacao,url_consulta,data_coleta,origem_pipeline
0,urnas eletrônicas,Lula perdeu todas as eleições com votação manu...,2025-12-07T00:00:00Z,AFP Checamos,https://checamos.afp.com/doc.afp.com.88868T3,Enganoso,2025-12-15T17:22:00Z,google_factcheck_raw.csv,NaN,NaN,NaN,NaN
1,urnas eletrônicas,Lula perdeu todas as eleições feitas com cédul...,2025-12-09T00:00:00Z,Aos Fatos,https://www.aosfatos.org/noticias/falso-que-lu...,falso,2025-12-09T00:00:00Z,google_factcheck_raw.csv,NaN,NaN,NaN,NaN
2,urnas eletrônicas,No congresso americano todas as urnas foram ha...,2025-11-02T00:00:00Z,Projeto Comprova,https://projetocomprova.com.br/publica%C3%A7%C...,Falso,2025-11-07T00:00:00Z,google_factcheck_raw.csv,NaN,NaN,NaN,NaN


## Limpeza e padronização

In [5]:
df = df_raw.copy()

# --- texto_principal ---
df["texto_principal"] = df["texto_afirmacao"].apply(remover_html)

# Descartar registros sem texto
antes = len(df)
df = df[df["texto_principal"].str.strip() != ""].copy()
print(f"Removidos sem texto: {antes - len(df)}")

# --- fonte normalizada ---
df["fonte"] = (
    df["fonte_verificacao"]
    .fillna("DESCONHECIDA")
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", "_", regex=True)
)

# --- datas padronizadas ---
df["data_publicacao"] = df["data_publicacao"].apply(padronizar_data)

# --- avaliação normalizada ---
df["avaliacao_original"] = df["avaliacao_original"].fillna("").str.strip()
df["avaliacao_categoria"] = df["avaliacao_original"].apply(mapear_avaliacao)

# --- url_origem ---
df["url_origem"] = df["url_checagem"].fillna("").str.strip()

print("Limpeza aplicada.")
print(f"\nDistribuição de avaliacao_categoria:")
print(df["avaliacao_categoria"].value_counts())

Removidos sem texto: 0
Limpeza aplicada.

Distribuição de avaliacao_categoria:
avaliacao_categoria
FALSO               3774
ENGANOSO            1222
OUTRO                614
FORA_DE_CONTEXTO     111
IMPRECISO              5
Name: count, dtype: int64


## Remoção de duplicatas

In [6]:
antes = len(df)
df = df.drop_duplicates(
    subset=["texto_principal", "fonte", "url_origem"],
    keep="first"
).copy()
print(f"Duplicatas removidas: {antes - len(df)}")
print(f"Registros únicos: {len(df)}")

Duplicatas removidas: 1963
Registros únicos: 3763


## Montagem do DataFrame curated

In [ ]:
df["id_registro"]      = [str(uuid.uuid4()) for _ in range(len(df))]
df["pipeline"]          = "google_factcheck"
df["tipo_conteudo"]     = "AFIRMACAO_CHECADA"
df["data_curadoria"]    = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# rotulo_preliminar reflete a avaliação do fact-checker:
# claims confirmados como verdadeiros recebem VERDADEIRO; demais recebem FALSO.
df["rotulo_preliminar"] = df["avaliacao_categoria"].apply(
    lambda cat: "VERDADEIRO" if cat == "VERDADEIRO" else "FALSO"
)

# Colunas obrigatórias do schema curated
COLUNAS_OBRIGATORIAS = [
    "id_registro",
    "texto_principal",
    "rotulo_preliminar",
    "pipeline",
    "fonte",
    "tipo_conteudo",
    "data_publicacao",
    "url_origem",
    "data_curadoria",
]

# Colunas de contexto específicas deste pipeline
COLUNAS_CONTEXTO = [
    "avaliacao_original",
    "avaliacao_categoria",
    "termo_busca",
    "arquivo_raw_origem",
]

df_curated = df[COLUNAS_OBRIGATORIAS + COLUNAS_CONTEXTO].reset_index(drop=True)

print(f"Shape final do curated: {df_curated.shape}")
print(f"\nColunas: {list(df_curated.columns)}")
df_curated.head(3)

## Verificação de qualidade

In [8]:
print("=== Verificação de qualidade ===")
print(f"\nTotal de registros: {len(df_curated)}")
print(f"\nValores nulos por coluna:")
print(df_curated[COLUNAS_OBRIGATORIAS].isnull().sum())
print(f"\nDistribuição por fonte:")
print(df_curated["fonte"].value_counts().head(15))
print(f"\nDistribuição por avaliacao_categoria:")
print(df_curated["avaliacao_categoria"].value_counts())
print(f"\nRegistros com data_publicacao preenchida: {(df_curated['data_publicacao'] != '').sum()}")

=== Verificação de qualidade ===

Total de registros: 3763

Valores nulos por coluna:
id_registro          0
texto_principal      0
rotulo_preliminar    0
pipeline             0
fonte                0
tipo_conteudo        0
data_publicacao      0
url_origem           0
data_curadoria       0
dtype: int64

Distribuição por fonte:
fonte
AOS_FATOS              945
ESTADÃO                754
UOL_NOTÍCIAS           557
AFP_CHECAMOS           533
BOATOS.ORG             291
PROJETO_COMPROVA       230
DESCONHECIDA           191
OBSERVADOR             123
BOL_-_UOL               61
FOLHA_-_UOL             58
METRÓPOLES               7
NEXO_JORNAL              5
AGÊNCIA_TATU             4
CORREIO_BRAZILIENSE      2
BOL                      1
Name: count, dtype: int64

Distribuição por avaliacao_categoria:
avaliacao_categoria
FALSO               2434
ENGANOSO             740
OUTRO                500
FORA_DE_CONTEXTO      84
IMPRECISO              5
Name: count, dtype: int64

Registros com data_pu

## Exportação para curated/

In [9]:
data_agora = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
caminho_saida = PASTA_CURATED / f"google_factcheck_curated_{data_agora}.csv"

df_curated.to_csv(caminho_saida, index=False, encoding="utf-8-sig")

print(f"Arquivo curated salvo em: {caminho_saida}")
print(f"Total de registros exportados: {len(df_curated)}")
print(f"Data e hora da curadoria: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")

Arquivo curated salvo em: ..\dados\pipeline_falso_google_factcheck\curated\google_factcheck_curated_2026-05-17_02-45-52.csv
Total de registros exportados: 3763
Data e hora da curadoria: 17/05/2026 02:45:52
